# R3-N2 LR Audio Inventory

This notebook inventories audio metadata on Google Drive for later latency-fixture review. It performs **no model inference**, does not create crops, and does not modify dataset files. A technically long enough recording is not automatically authorized for benchmarking. Test recordings are not opened for audio metadata; unknown partition membership remains unknown and is not promoted as eligible.

Run in Google Colab. Review the partition/provenance status before using any candidate. Transcript contents are never read.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import csv, json, os, re, traceback
import pandas as pd


ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')
DATASET = ROOT / 'dataset'
OUTPUT = ROOT / 'reports/r3'

EXPECTED = {
    ('genuine', None): {
        'audio_dir': DATASET / 'genuine/wav',
        'transcript_dir': DATASET / 'genuine/transcripts',
    },
    ('tts', 'ChatterboxTTS'): {
        'audio_dir': DATASET / 'tts/ChatterboxTTS',
        'transcript_dir': None,
    },
    ('tts', 'MeloTTS'): {
        'audio_dir': DATASET / 'tts/MeloTTS',
        'transcript_dir': None,
    },
    ('tts', 'VITS'): {
        'audio_dir': DATASET / 'tts/VITS',
        'transcript_dir': None,
    },
    ('tts', 'XTTSv2'): {
        'audio_dir': DATASET / 'tts/XTTSv2',
        'transcript_dir': None,
    },
}

TARGET_SECONDS = (1, 4, 10, 30, 60)

if not ROOT.is_dir():
    raise FileNotFoundError(f'Project root not found: {ROOT}')

missing_directories = []

for (category, system), config in EXPECTED.items():
    audio_dir = config['audio_dir']
    transcript_dir = config['transcript_dir']

    if not audio_dir.is_dir():
        missing_directories.append({
            'kind': 'audio',
            'category': category,
            'tts_system': system,
            'path': str(audio_dir),
        })

    if transcript_dir is not None and not transcript_dir.is_dir():
        missing_directories.append({
            'kind': 'transcript',
            'category': category,
            'tts_system': system,
            'path': str(transcript_dir),
        })

print('Expected directories missing:', len(missing_directories))

for row in missing_directories:
    print(row['path'])


Expected directories missing: 0


## Partition and metadata discovery

The notebook first uses explicit partition/split words in path components, then inspects files whose names indicate split/partition/manifest metadata. It only interprets metadata tables when recognizable path/filename and partition columns are present. Conflicting or unmatched evidence stays `unknown` (or `conflict`). A recording identified as test is not passed to audio metadata readers. This generic matching does not establish dataset approval; review the evidence before benchmarking.

In [3]:
PARTITION_TOKENS = {
    'train': 'train', 'training': 'train', 'validation': 'validation', 'valid': 'validation',
    'val': 'validation', 'test': 'test', 'testing': 'test', 'calibration': 'calibration',
    'operating': 'operating'
}
PATH_COLUMNS = {'path', 'filepath', 'file_path', 'relative_path', 'audio_path', 'recording_path', 'filename', 'file_name'}
PARTITION_COLUMNS = {'partition', 'split', 'subset', 'role', 'set'}

def norm_path(value):
    s = str(value).strip().replace('\\', '/').lstrip('./')
    return re.sub('/+', '/', s).casefold()

def partition_from_value(value):
    tokens = re.findall(r'[a-z]+', str(value).casefold())
    found = {PARTITION_TOKENS[t] for t in tokens if t in PARTITION_TOKENS}
    return next(iter(found)) if len(found) == 1 else ('conflict' if len(found) > 1 else None)

# Discover metadata by filename only first; transcript paths are excluded.
metadata_paths = []
if DATASET.is_dir():
    for p in DATASET.rglob('*'):
        if not p.is_file() or any('transcript' in part.casefold() for part in p.relative_to(DATASET).parts[:-1]):
            continue
        if re.search(r'(split|partition|manifest)', p.name, re.I) and p.suffix.casefold() in {'.csv', '.tsv', '.parquet', '.json'}:
            metadata_paths.append(p)
metadata_inventory = []
metadata_maps = {}
for p in metadata_paths:
    item = {'path': str(p.relative_to(ROOT)), 'format': p.suffix.casefold(), 'status': 'discovered', 'columns': []}
    try:
        if p.suffix.casefold() == '.csv': df = pd.read_csv(p, dtype=str)
        elif p.suffix.casefold() == '.tsv': df = pd.read_csv(p, sep='\t', dtype=str)
        elif p.suffix.casefold() == '.parquet': df = pd.read_parquet(p)
        else:
            obj = json.loads(p.read_text(encoding='utf-8'))
            if isinstance(obj, list): df = pd.json_normalize(obj)
            elif isinstance(obj, dict):
                rows = next((v for v in obj.values() if isinstance(v, list) and (not v or isinstance(v[0], dict))), None)
                df = pd.json_normalize(rows) if rows is not None else pd.json_normalize(obj)
            else: df = pd.DataFrame()
        item['columns'] = [str(c) for c in df.columns]
        part_cols = [c for c in df.columns if str(c).casefold() in PARTITION_COLUMNS]
        path_cols = [c for c in df.columns if str(c).casefold() in PATH_COLUMNS]
        if part_cols and path_cols:
            item['status'] = 'partition columns recognized; exact path/name matching attempted'
            for _, row in df.iterrows():
                labels = {partition_from_value(row[c]) for c in part_cols}
                labels.discard(None)
                label = next(iter(labels)) if len(labels) == 1 else ('conflict' if labels else None)
                if label is None: continue
                for c in path_cols:
                    val = row.get(c)
                    if pd.isna(val): continue
                    key = norm_path(val)
                    metadata_maps.setdefault(key, set()).add(label)
        else:
            item['status'] = 'no recognizable path/partition column pair; not used for assignment'
    except Exception as exc:
        item['status'] = f'unreadable metadata: {type(exc).__name__}: {exc}'
    metadata_inventory.append(item)
print('Partition metadata files discovered:', len(metadata_inventory))
for item in metadata_inventory: print(item)

Partition metadata files discovered: 0


## Recording inventory

Audio duration/sample-rate/channel metadata is obtained with `soundfile.info` (header-level inspection). The standard-library WAV header reader is a fallback. Files marked `test` are not opened. Where partition remains unknown, audio metadata may be read for inventory but the row is never marked as an approved benchmark candidate. No audio samples are decoded into arrays.

In [5]:
def infer_partition(path, audio_root):
    rel_dataset = path.relative_to(DATASET)
    path_labels = {PARTITION_TOKENS[t.casefold()] for t in rel_dataset.parts[:-1] if t.casefold() in PARTITION_TOKENS}
    evidence = []
    if path_labels:
        evidence.append(next(iter(path_labels)) if len(path_labels) == 1 else 'conflict')
    # Match metadata values against dataset-relative path, audio-root-relative path, and basename.
    keys = {norm_path(rel_dataset), norm_path(path.relative_to(audio_root)), norm_path(path.name)}
    for key, labels in metadata_maps.items():
        if key in keys or any(key.endswith('/' + k) or k.endswith('/' + key) for k in keys if key and k):
            evidence.extend(labels)
    unique = set(evidence)
    if len(unique) == 1: return next(iter(unique)), 'path or metadata match'
    if len(unique) > 1 or 'conflict' in unique: return 'conflict', 'conflicting path/metadata evidence'
    return 'unknown', 'no partition evidence found'

def inspect_audio(path):
    try:
        import soundfile as sf
        info = sf.info(str(path))
        return float(info.duration), int(info.samplerate), int(info.channels), None
    except Exception as first_error:
        try:
            import wave
            with wave.open(str(path), 'rb') as w:
                frames, rate, channels = w.getnframes(), w.getframerate(), w.getnchannels()
                if rate <= 0: raise ValueError('invalid WAV sample rate')
                return frames / rate, rate, channels, None
        except Exception as exc:
            return None, None, None, f'{type(exc).__name__}: {exc}; soundfile: {type(first_error).__name__}: {first_error}'

EXPECTED = {
    key: (Path(audio_dir), Path(transcript_dir))
    for key, (audio_dir, transcript_dir) in EXPECTED.items()
}

transcript_index = {}
for (category, system), (audio_dir, transcript_dir) in EXPECTED.items():
    if transcript_dir.is_dir():
        for p in transcript_dir.rglob('*'):
            if p.is_file(): transcript_index.setdefault((category, system), set()).add(p.stem.casefold())

rows = []
for (category, system), (audio_dir, transcript_dir) in EXPECTED.items():
    if not audio_dir.is_dir(): continue
    for p in sorted(audio_dir.rglob('*')):
        if not p.is_file() or p.suffix.casefold() != '.wav': continue
        partition, partition_evidence = infer_partition(p, audio_dir)
        size = p.stat().st_size
        duration = rate = channels = None
        error = None
        if partition == 'test':
            error = 'audio metadata intentionally not read: path/metadata identifies test partition'
        else:
            duration, rate, channels, error = inspect_audio(p)
        rel = p.relative_to(ROOT).as_posix()
        transcript_match = p.stem.casefold() in transcript_index.get((category, system), set())
        row = {
            'relative_drive_path': rel, 'source_category': category, 'tts_system': system or '',
            'filename': p.name, 'duration_seconds': duration, 'sample_rate_hz': rate,
            'channels': channels, 'file_size_bytes': size, 'transcript_stem_match': transcript_match,
            'partition_status': partition, 'partition_evidence': partition_evidence,
            'benchmark_authorized': False, 'metadata_error': error or ''
        }
        for target in TARGET_SECONDS:
            row[f'can_supply_{target}s_prefix'] = bool(duration is not None and duration >= target)
        rows.append(row)

inventory = pd.DataFrame(rows)
if inventory.empty:
    inventory = pd.DataFrame(columns=['relative_drive_path','source_category','tts_system','filename','duration_seconds','sample_rate_hz','channels','file_size_bytes','transcript_stem_match','partition_status','partition_evidence','benchmark_authorized','metadata_error'] + [f'can_supply_{t}s_prefix' for t in TARGET_SECONDS])
print('WAV recordings inventoried:', len(inventory))
display(inventory)

WAV recordings inventoried: 0


,relative_drive_path,source_category,tts_system,filename,duration_seconds,sample_rate_hz,channels,file_size_bytes,transcript_stem_match,partition_status,partition_evidence,benchmark_authorized,metadata_error,can_supply_1s_prefix,can_supply_4s_prefix,can_supply_10s_prefix,can_supply_30s_prefix,can_supply_60s_prefix


## Summary, candidate review, and reports

`benchmark_authorized` deliberately remains false for every row: this inventory cannot grant approval. Counts of technically duration-eligible recordings are informational and include a separate partition breakdown. A candidate is usable only after its partition and authorization have been independently confirmed. Files are written under the requested Drive project report folder, with unique UTC timestamps.

In [6]:
category_counts = (inventory.groupby(['source_category','tts_system'], dropna=False).size().rename('recordings').reset_index() if len(inventory) else pd.DataFrame(columns=['source_category','tts_system','recordings']))
print('\nCounts by source category and TTS system:')
display(category_counts)

if len(inventory) and inventory['duration_seconds'].notna().any():
    duration_distribution = inventory.loc[inventory.duration_seconds.notna(), 'duration_seconds'].describe(percentiles=[.1,.25,.5,.75,.9]).to_frame('duration_seconds')
else:
    duration_distribution = pd.DataFrame({'duration_seconds': []})
print('Duration distribution (seconds):')
display(duration_distribution)

print('Missing expected directories:')
display(pd.DataFrame(missing_directories))
unreadable = inventory[inventory.metadata_error.ne('')].copy() if len(inventory) else inventory.copy()
print('Missing/unreadable/metadata-skipped files:')
display(unreadable)

eligibility_rows = []
for target in TARGET_SECONDS:
    col = f'can_supply_{target}s_prefix'
    eligible = inventory[inventory[col]] if len(inventory) else inventory
    for partition in ['train','validation','calibration','operating','test','unknown','conflict']:
        n = int((eligible.partition_status == partition).sum()) if len(eligible) else 0
        eligibility_rows.append({'target_seconds': target, 'partition_status': partition, 'duration_eligible_recordings': n, 'authorization_confirmed': 0})
eligibility = pd.DataFrame(eligibility_rows)
print('Technically duration-eligible counts (not authorization):')
display(eligibility)

OUTPUT.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
csv_path = OUTPUT / f'r3_n2_lr_audio_inventory_{run_id}.csv'
json_path = OUTPUT / f'r3_n2_lr_audio_inventory_{run_id}.json'
inventory.to_csv(csv_path, index=False, quoting=csv.QUOTE_MINIMAL)
report = {
    'schema_version': 'r3-n2-lr-audio-inventory-v1', 'run_id': run_id,
    'created_utc': datetime.now(timezone.utc).isoformat(), 'project_root': str(ROOT),
    'dataset_root': str(DATASET), 'expected_directories_missing': missing_directories,
    'metadata_files_inspected': metadata_inventory,
    'counts_by_category_and_tts_system': category_counts.fillna('').to_dict(orient='records'),
    'duration_distribution_seconds': duration_distribution.reset_index().rename(columns={'index':'statistic'}).to_dict(orient='records'),
    'duration_eligibility': eligibility.to_dict(orient='records'),
    'partition_status_counts': inventory.partition_status.value_counts(dropna=False).to_dict() if len(inventory) else {},
    'unreadable_or_skipped_count': int(len(unreadable)),
    'inventory_csv': str(csv_path),
    'safety': {'audio_inference_run': False, 'benchmark_run': False, 'crops_created': False, 'test_audio_opened': False, 'transcript_contents_read': False, 'dataset_files_modified': False, 'benchmark_authorization_granted': False, 'unknown_partition_is_eligible': False}
}
json_path.write_text(json.dumps(report, indent=2, ensure_ascii=False, default=str) + '\n', encoding='utf-8')
print('Saved CSV:', csv_path)
print('Saved JSON:', json_path)
print('Inventory completion describes this Colab run only; no benchmark authorization is inferred.')


Counts by source category and TTS system:


,source_category,tts_system,recordings


Duration distribution (seconds):


,duration_seconds


Missing expected directories:


""


Missing/unreadable/metadata-skipped files:


,relative_drive_path,source_category,tts_system,filename,duration_seconds,sample_rate_hz,channels,file_size_bytes,transcript_stem_match,partition_status,partition_evidence,benchmark_authorized,metadata_error,can_supply_1s_prefix,can_supply_4s_prefix,can_supply_10s_prefix,can_supply_30s_prefix,can_supply_60s_prefix


Technically duration-eligible counts (not authorization):


,target_seconds,partition_status,duration_eligible_recordings,authorization_confirmed
0,1,train,0,0
1,1,validation,0,0
2,1,calibration,0,0
3,1,operating,0,0
4,1,test,0,0
5,1,unknown,0,0
6,1,conflict,0,0
7,4,train,0,0
8,4,validation,0,0
9,4,calibration,0,0


Saved CSV: /content/drive/MyDrive/PAI&AIES Project/reports/r3/r3_n2_lr_audio_inventory_20261114T052853Z.csv
Saved JSON: /content/drive/MyDrive/PAI&AIES Project/reports/r3/r3_n2_lr_audio_inventory_20261114T052853Z.json
Inventory completion describes this Colab run only; no benchmark authorization is inferred.


In [7]:

from pathlib import Path
import csv
import soundfile as sf

ROOT = Path("/content/drive/MyDrive/PAI&AIES Project")
DATASET = ROOT / "dataset"

TARGETS = {
    1: (1, 2),
    4: (4, 5),
    10: (10, 11),
    30: (30, 31),
    60: (60, 61),
}

SOURCES = {
    "genuine": DATASET / "genuine/wav",
    "chatterbox": DATASET / "tts/ChatterboxTTS",
    "melo": DATASET / "tts/MeloTTS",
    "vits": DATASET / "tts/VITS",
    "xtts": DATASET / "tts/XTTSv2",
}

rows = []

for system, folder in SOURCES.items():
    if not folder.is_dir():
        print("Missing:", folder)
        continue

    for path in folder.rglob("*.wav"):
        try:
            info = sf.info(str(path))
            duration = info.duration

            for target, (low, high) in TARGETS.items():
                if low <= duration <= high:
                    rows.append({
                        "system": system,
                        "target_seconds": target,
                        "duration_seconds": round(duration, 3),
                        "sample_rate": info.samplerate,
                        "channels": info.channels,
                        "path": str(path),
                    })
        except Exception as exc:
            print("Skipped:", path.name, exc)

output = ROOT / "reports/r3/fixture_candidates.csv"
output.parent.mkdir(parents=True, exist_ok=True)

with output.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys() if rows else [
        "system", "target_seconds", "duration_seconds",
        "sample_rate", "channels", "path"
    ])
    writer.writeheader()
    writer.writerows(rows)

print("Candidate recordings:", len(rows))
print("Saved:", output)


KeyboardInterrupt: 